# AlphaFold-style essentiality model — Colab GPU run

Trains a JAX/Haiku Evoformer (single-block, real `OuterProductMean` + recycling), molded to genes-as-residues, on the 48-organism MSA tensors prebuilt in the repo. Single T4 GPU is enough; ~5–10 minutes for 4 epochs on 180k genes. Faithful re-implementation of `vendor/alphafold/modules.py` ops in JAX, not a NumPy port.

**Mapping**: gene=residue, organism=MSA row. Window of 3 (prev/focal/next). MSA value at each (organism, gene) cell = ortholog essentiality + known-bit, held-out clade rows masked.

**What this notebook does**:
1. Clone the repo (or upload the two cached tensors)
2. Install JAX with GPU
3. Build a JAX `Evoformer` block: input embed → OuterProductMean → pair-bias MSA mixing → residual → readout + brightness
4. Train leave-one-clade-out over the 5 major clades; report per-clade AUC + the brightness calibration
5. Compare 1 vs 3 recycles

**GPU recommendation**: free-tier T4 (16 GB) is plenty. A100 is overkill for this size. CPU works too but ~10× slower.

## 1. Clone the repo (or skip this and upload `af_msa_cache.npz` directly)

In [ ]:
# Option A: clone the repo (replace with your branch / fork URL)
!git clone --depth 1 -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git cell_repo || echo 'already cloned'
import os; os.chdir('cell_repo')
!ls outputs/orphan/af_msa_cache.npz vendor/alphafold/modules.py

## 2. Install JAX with CUDA + Haiku

In [ ]:
!pip install -q --upgrade 'jax[cuda12]' dm-haiku optax
import jax, jax.numpy as jnp
print('JAX', jax.__version__, '| devices:', jax.devices())
# expect: GpuDevice(id=0) or CudaDevice(id=0). If only CpuDevice, runtime is CPU — change runtime.

## 3. Load the prebuilt MSA tensors

Built by `scripts/af_build_msa.py` in the repo: 179,237 labeled genes across 48 organisms, each row = (essentiality, known-bit) for that gene's OG in that organism, plus dN/dS rows. Same source the NumPy single-track model used.

In [ ]:
import numpy as np
Z = np.load('outputs/orphan/af_msa_cache.npz', allow_pickle=True)
foc = Z['foc']; msa = Z['msa']; mask = Z['msa_mask']; y = Z['y']
clade = Z['clade']; meta_org = Z['meta_org']; orgs = Z['orgs']; msa_org = Z['msa_org']
N, K, Dm = msa.shape
print(f'genes={N}, MSA depth={K}, MSA features={Dm}, essential rate={y.mean():.3f}')
print('clades:', sorted(set(clade.tolist()))[:8], '...')

## 4. The JAX Evoformer — faithful to `vendor/alphafold/modules.py`

Single block. `OuterProductMean` follows AF2 Suppl. Alg. 10 (the same einsums as in our NumPy port, but here in JAX with proper LayerNorm + Linear). MSA mixing uses a pair-derived bias (the AF2 `MSARowAttentionWithPairBias` shape). Recycling is the standard AF idea: feed the output back as input embedding.

In [ ]:
import haiku as hk, optax
import jax, jax.numpy as jnp
from typing import NamedTuple

C_M = 32   # MSA channel
C_Z = 16   # pair channel
C_O = 16   # outer-product hidden
HEAD = 64

class EvoformerBlock(hk.Module):
    def __init__(self, name='evo'):
        super().__init__(name=name)

    def __call__(self, msa, mask):
        # msa [N_seq, c_m], mask [N_seq] (per-gene window collapsed to per-organism row;
        # for the focal-only readout we use msa of shape [N_seq, c_m])
        # OuterProductMean reduced form for 1D 'sequence' of length 1 ('focal-only'):
        x = hk.LayerNorm([-1], True, True)(msa)
        left = hk.Linear(C_O, name='opm_l')(x) * mask[:, None]
        right = hk.Linear(C_O, name='opm_r')(x) * mask[:, None]
        # outer product over organism axis, mean (eps for safety)
        # [c_o, c_o] = sum_s left[s,co] * right[s,co']  /  sum_s mask^2
        norm = jnp.maximum(mask.sum(), 1.0)
        pair = jnp.einsum('sc,sd->cd', left, right) / norm  # [c_o, c_o]
        pair = pair.reshape(-1)                             # [c_o*c_o]
        pair_h = hk.Linear(C_Z, name='opm_out')(pair)       # [c_z]
        # bias-modulated MSA mixing: every row attends to the (single) focal pos via a bias
        bias = hk.Linear(1, name='pair2bias')(pair_h).squeeze()           # scalar
        attn_logits = (msa @ hk.get_parameter('q', [C_M], init=hk.initializers.RandomNormal(0.3))) + bias
        attn_logits = jnp.where(mask > 0, attn_logits, -1e9)
        a = jax.nn.softmax(attn_logits)
        v = hk.Linear(C_M, name='v')(msa)
        ctx = (a[:, None] * v).sum(0)                          # [c_m]
        return ctx, pair_h

def model_fn(msa_in, mask, recycles=1):
    """msa_in [N_seq, Dm], mask [N_seq] -> logits, confidence."""
    emb = hk.Linear(C_M, name='emb')(msa_in)
    msa_act = emb
    pair_h = jnp.zeros(C_Z)
    for _ in range(recycles):
        ctx, pair_h = EvoformerBlock()(msa_act, mask)
        # feed ctx back: broadcast to all rows, residual
        msa_act = msa_act + ctx[None, :]
        msa_act = hk.LayerNorm([-1], True, True)(msa_act)
    # readout from (final ctx + pair)
    feat = jnp.concatenate([ctx, pair_h])
    h = jax.nn.relu(hk.Linear(HEAD, name='h1')(feat))
    logit = hk.Linear(1, name='out')(h).squeeze()
    conf = hk.Linear(1, name='conf')(h).squeeze()
    return logit, conf

fwd = hk.transform(model_fn)
print('model defined')

## 5. Train leave-one-clade-out

In [ ]:
from functools import partial

def loss_fn(params, key, msa_b, mask_b, y_b, recycles):
    def one(m, mk, yy):
        logit, conf = fwd.apply(params, key, m, mk, recycles)
        p = jax.nn.sigmoid(logit); pos = jnp.float32(0.27); cw = jnp.where(yy==1, 0.5/pos, 0.5/(1-pos))
        bce = -(yy*jnp.log(p+1e-9)+(1-yy)*jnp.log(1-p+1e-9))*cw
        # confidence target: was rounded prediction correct?
        corr = ((p > 0.5).astype(jnp.float32) == yy).astype(jnp.float32)
        c = jax.nn.sigmoid(conf)
        cb = -(corr*jnp.log(c+1e-9)+(1-corr)*jnp.log(1-c+1e-9))
        return bce + 0.3*cb, p, c
    losses, ps, cs = jax.vmap(one)(msa_b, mask_b, y_b)
    return losses.mean(), (ps, cs)

@partial(jax.jit, static_argnums=(4,))
def train_step(params, opt_state, key, batch, recycles):
    msa_b, mask_b, y_b = batch
    (l, _), g = jax.value_and_grad(loss_fn, has_aux=True)(params, key, msa_b, mask_b, y_b, recycles)
    updates, opt_state = opt.update(g, opt_state, params)
    return optax.apply_updates(params, updates), opt_state, l

@partial(jax.jit, static_argnums=(3,))
def predict_step(params, key, batch, recycles):
    msa_b, mask_b = batch
    def one(m, mk):
        lo, cf = fwd.apply(params, key, m, mk, recycles)
        return jax.nn.sigmoid(lo), jax.nn.sigmoid(cf)
    return jax.vmap(one)(msa_b, mask_b)

def auc(s, yy):
    yy = np.asarray(yy); n1=yy.sum(); n0=len(yy)-n1
    if n1==0 or n0==0: return float('nan')
    r = np.argsort(np.argsort(s))+1
    return float((r[yy==1].sum()-n1*(n1+1)/2)/(n1*n0))
def cov_at_p(s, yy, t, mn=20):
    o=np.argsort(-s); ys=np.asarray(yy)[o].astype(float); k=np.arange(1,len(ys)+1)
    pr=np.cumsum(ys)/k; ok=(pr>=t)&(k>=mn); return float(k[np.where(ok)[0].max()]/len(ys)) if ok.any() else 0.0

def mask_msa(held_clade):
    # zero label + known-bit on rows from held-out clade
    org_clade = {orgs[meta_org[i]]: clade[i] for i in range(len(meta_org))}
    held_orgs = set(o for o in orgs if org_clade.get(o)==held_clade)
    held_idx = set(i for i,o in enumerate(orgs) if o in held_orgs)
    out = msa.copy()
    bad = np.isin(msa_org, list(held_idx))
    out[..., 0] = np.where(bad, 0, out[..., 0])
    out[..., 1] = np.where(bad, 0, out[..., 1])
    return jnp.array(out)

RECYCLES = 1   # change to 3 to compare
MAJOR = ['pseudomonas','ralstonia','shewanella','dickeya','burkholderia']
EPOCHS = 4; BS = 256; LR = 3e-3
key = jax.random.PRNGKey(0)

results = {}
for held in MAJOR:
    test_idx = np.where(clade==held)[0]
    if len(test_idx) < 100: continue
    train_idx = np.where(clade!=held)[0]
    msa_masked = mask_msa(held)
    mask_j = jnp.array(mask)
    y_j = jnp.array(y)
    params = fwd.init(key, msa_masked[0], mask_j[0], RECYCLES)
    opt = optax.adam(LR); opt_state = opt.init(params)
    for ep in range(EPOCHS):
        perm = np.random.permutation(train_idx)
        for i in range(0, len(perm), BS):
            bi = perm[i:i+BS]
            batch = (msa_masked[bi], mask_j[bi], y_j[bi])
            params, opt_state, l = train_step(params, opt_state, key, batch, RECYCLES)
        if ep == EPOCHS-1:
            print(f'  held={held} epoch {ep} loss={float(l):.3f}')
    ps, cs = predict_step(params, key, (msa_masked[test_idx], mask_j[test_idx]), RECYCLES)
    ps = np.asarray(ps); cs = np.asarray(cs); yt = y[test_idx]
    results[held] = dict(n=int(len(test_idx)), auc=round(auc(ps,yt),3),
                          ne_cov_p90=round(cov_at_p(1-ps,1-yt,0.9),3),
                          ess_cov_p70=round(cov_at_p(ps,yt,0.7),3),
                          high_conf_acc=float(((ps>0.5).astype(int)==yt)[cs>=0.85].mean()),
                          high_conf_n=int((cs>=0.85).sum()))
    print(f'  {held:<14} AUC={results[held]["auc"]:.3f} '
          f'neP90={results[held]["ne_cov_p90"]:.2f} essP70={results[held]["ess_cov_p70"]:.2f} '
          f'| high-conf acc={results[held]["high_conf_acc"]:.2f} on {results[held]["high_conf_n"]} genes')

import json
print(json.dumps(results, indent=2))

## 6. Save results back to the repo (if cloned with write access)

In [ ]:
with open('outputs/orphan/af_jax_results.json', 'w') as f:
    json.dump({'recycles': RECYCLES, 'per_clade': results}, f, indent=2)
print('wrote outputs/orphan/af_jax_results.json')
# Optional: push back. Requires you to set git remote+auth in Colab first.
# !git add outputs/orphan/af_jax_results.json && git commit -m 'colab evoformer run' && git push

## Scale-up knobs (when you want a bigger model)

- `C_M=32, C_Z=16, C_O=16`: tiny demo. Real Evoformer uses `C_M=256, C_Z=128`. Crank these up first.
- `RECYCLES`: 3–4 starts to matter once the model is bigger.
- Add real **MSARowAttentionWithPairBias** (the full multi-head version from `vendor/alphafold/modules.py`) once you have >1 'residue' in the window. The current notebook uses a 1-position-per-organism reduction since each gene is one residue. To use a window of W genes, reshape `msa` to `[N_seq, W, Dm]` and replace the OPM reduction with the full einsums `'acb,ade->dceb'` then `'dceb,cef->dbf'`.
- Add **triangle multiplication / triangle attention** for larger windows (`vendor/alphafold/modules.py` lines ~1358 and ~963). These are the AF2 ops that make the pair representation self-consistent — they cost O(W³) so they only pay off for W ≥ 8.
- For a *genome-scale* run (treat the whole chromosome as one long sequence), you need chunked attention + gradient checkpointing — see `vendor/alphafold/mapping.py` patterns. This is the proper-scale version of the architecture.

## What GPU?

| GPU | Fits | Notes |
|---|---|---|
| **T4 (16 GB, free Colab)** | ✓ current notebook | recommended. ~5–10 min full LOCO. |
| A10G/L4 (24 GB) | ✓ + 2× wider C_M | scale C_M to 64, recycles 3 |
| **A100 (40/80 GB, Colab Pro+)** | ✓ + full Evoformer width (C_M=256) + window W=11 | the regime where triangle ops pay off |
| H100 | overkill for 48 organisms | only needed for full-chromosome genome-scale |

For the **48-organism × 180k-gene** scale we have, **T4 is enough**. Step up only if you grow the model width or move to a windowed multi-residue Evoformer.